Osservando il grafico prodotto nella lezione, noterai che la curva rossa (predetta) sembra spesso 'inseguire' qualla verde con un leggero ritardo (lag). Questo accade perchè il modello tende a dare molto peso all'ultimo valore visto. Modifica dell'architettura: inserisce un layer Conv1D prima della LSTM. Questo permetterà al modello di estrarre feature locali (trend di breve termine) prima di passare alla memoria sequenziale della LSTM. Aumento della finestra: porta la WINDOW_SIZE a 365 (un intero anno). Cosa succede al tempo di addestramento? e la precisione sui picchi stagionali miglioare?. Analisi del residuo: Calcola e stampa l'errore medio assoluto (MAE) tra actual e predicted.

In [ ]:
import os

# 1. CONFIGURAZIONE AMBIENTE
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["KERAS_BACKEND"] = "torch"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import tensorflow as tf
import keras
from keras import layers, utils
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 2. CARICAMENTO E PRE-ELABORAZIONE DATI
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
df = pd.read_csv(url, parse_dates=['Date'], index_col='Date')

values = df.values.astype('float32')
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_values = scaler.fit_transform(values)

# --- MODIFICHE RICHIESTE ---
WINDOW_SIZE = 365  # Finestra annuale per catturare trend stagionali
BATCH_SIZE = 512   
SPLIT_TIME = 2500 

train_series = scaled_values[:SPLIT_TIME]
test_series = scaled_values[SPLIT_TIME:]

# ---------------------------------------------------------
# 3. DATA PIPELINE (FIX PER IL WARNING CACHE)
# ---------------------------------------------------------
def create_dataset(series, window_size, batch_size, is_train=True):
    dataset = utils.timeseries_dataset_from_array(
        data=series[:-1],
        targets=series[window_size:],
        sequence_length=window_size,
        batch_size=batch_size,
        shuffle=is_train
    )
    # Spostiamo il prefetch DOPO la cache e usiamo l'autotune di TF
    # Questo assicura che il dataset sia completamente letto prima di essere messo in cache
    dataset = dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

train_set = create_dataset(train_series, WINDOW_SIZE, BATCH_SIZE)
test_set = create_dataset(test_series, WINDOW_SIZE, BATCH_SIZE, is_train=False)

# ---------------------------------------------------------
# 4. ARCHITETTURA CON CONV1D (Per ridurre il Lag)
# ---------------------------------------------------------


inputs = keras.Input(shape=(WINDOW_SIZE, 1))

# Modifica: Layer convoluzionale per estrarre trend locali prima della memoria LSTM
x = layers.Conv1D(filters=64, kernel_size=3, padding="causal", activation="relu")(inputs)

x = layers.LSTM(64, return_sequences=True)(x)
x = layers.Dropout(0.1)(x)
x = layers.LSTM(32)(x)
outputs = layers.Dense(1)(x)

model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=keras.optimizers.AdamW(learning_rate=1e-3), loss="mse", metrics=["mae"])

# 5. ADDESTRAMENTO
print(f"Esecuzione su backend: {keras.backend.backend()}")
callbacks = [keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)]

# Noterai che con 365 giorni il tempo per epoca è maggiore rispetto ai 30 giorni
history = model.fit(train_set, epochs=30, callbacks=callbacks, verbose=1)

# ---------------------------------------------------------
# 6. INFERENZA, DENORMALIZZAZIONE E ANALISI RESIDUI
# ---------------------------------------------------------
forecast_scaled = model.predict(test_set)
predicted = scaler.inverse_transform(forecast_scaled)
actual = values[SPLIT_TIME + WINDOW_SIZE:]

# Calcolo MAE (Analisi del Residuo richiesta)
mae_val = mean_absolute_error(actual, predicted)
print(f"\n--- PERFORMANCE METRICS ---")
print(f"Mean Absolute Error (MAE): {mae_val:.4f} °C")

# 7. VISUALIZZAZIONE


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))

# Plot Previsione
ax1.plot(actual[:250], label="Reale", color='#2A9D8F', alpha=0.7)
ax1.plot(predicted[:250], label="Predetto (Conv1D+LSTM)", color='#E76F51', linestyle='--')
ax1.set_title(f"Previsione con Finestra di 365 giorni (MAE: {mae_val:.2f}°C)")
ax1.legend()

# Plot Residui (Errore puntuale)
residuals = actual - predicted
ax2.scatter(range(len(residuals)), residuals, alpha=0.3, color='#457B9D', s=10)
ax2.axhline(0, color='black', linestyle='--')
ax2.set_title("Analisi dei Residui (Valore Reale - Predetto)")

plt.tight_layout()
plt.show()

Esecuzione su backend: torch
Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 218s 43s/step - loss: 0.0842 - mae: 0.2429
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 170s 29s/step - loss: 0.0305 - mae: 0.1481
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 151s 29s/step - loss: 0.0152 - mae: 0.0961
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 177s 33s/step - loss: 0.0183 - mae: 0.1064
Epoch 5/30
2/5 ━━━━━━━━━━━━━━━━━━━━ 1:43 35s/step - loss: 0.0143 - mae: 0.0927